## Table of Contents

- [Resources](#resources)

- [Where A2A Fits](#where-a2a-fits)

- [A2A Architecture](#a2a-architecture)

  - [Execution Modes](#execution-modes)

    - [Synchronous Communication](#synchronous-communication)

    - [Asynchronous Communication](#asynchronous-communication)

    - [Streaming using Server-Side Events (SSE)](#streaming-using-server-side-events-sse)

    - [Push Notification](#push-notification)

  - [Summary — Choosing the Right Mode](#summary--choosing-the-right-mode)

  - [AgentCard Capabilities — Multiple Skills with Different Modes](#agentcard-capabilities--multiple-skills-with-different-modes)

- [LangGraph Agent calling an A2A Server](#langgraph-agent-calling-an-a2a-server)

  - [How LangGraph Agent Invokes an A2A Server](#how-langgraph-agent-invokes-an-a2a-server)

  - [How AgentCard is Passed to LLM to Decide Which Skill to Call](#how-agentcard-is-passed-to-llm-to-decide-which-skill-to-call)

  - [How LangGraph Agent Calls a Specific Skill on an A2A Server](#how-langgraph-agent-calls-a-specific-skill-on-an-a2a-server)

- [Agent Stack](#agent-stack)

    - [From Local Development to Production Deployment](#from-local-development-to-production-deployment)

    - [Other Agentic Platform Options](#other-agentic-platform-options-and-why-they-fall-short)

    - [What Agent Stack Is](#what-agent-stack-is)

    - [Who Is Agent Stack For?](#who-is-agent-stack-for)

    - [Architecture](#architecture)

- [Security](#security)

- [Extension](#extension)

- [Observability](#observability)

---

# Resources

| Topic | URL | Description |

|-------|-----|-------------|

| A2A Official Docs | [a2a-protocol.org](https://a2a-protocol.org/latest/) | Official A2A protocol specification and latest documentation |

| A2A Summit 2025 Slides | [AI Agent Discovery Slides (PDF)](https://github.com/muscariello/a2a-summit-2025/blob/main/slides/ai-agent-discovery-slides.pdf) | Conference slides on AI agent discovery from the A2A Summit 2025 |

| DeepLearning.AI Course | [A2A: The Agent2Agent Protocol](https://learn.deeplearning.ai/courses/a2a-the-agent2agent-protocol) | Full course on building with the A2A protocol by DeepLearning.AI |

| Course Repos & Resources | [Course Repos & Resources](https://learn.deeplearning.ai/courses/a2a-the-agent2agent-protocol/lesson/79m1rh/course-repos-%26-resources) | GitHub repos and supplementary resources linked from the DeepLearning.AI course |

| Healthcare Agent (Example) | [AgentStack-HealthcareAgent](https://github.com/sandijean90/AgentStack-HealthcareAgent/tree/main) | A real-world A2A example: healthcare scheduling agent built with AgentStack |

| Tourist Scheduling (Example) | [tourist_scheduling_system](https://github.com/agntcy/agentic-apps/tree/main/tourist_scheduling_system) | A multi-agent tourist scheduling app demonstrating A2A communication patterns |

| AgentStack Starter | [agentstack-starter](https://github.com/i-am-bee/agentstack-starter) | Starter template to quickly bootstrap an A2A-compatible agent using BeeAI AgentStack |

| AgentStack Framework | [agentstack (GitHub)](https://github.com/i-am-bee/agentstack) | BeeAI's open-source AgentStack framework for building A2A-compliant agents |

| AgentStack Docs | [agentstack.beeai.dev](https://agentstack.beeai.dev/stable/introduction/welcome) | Official documentation for BeeAI AgentStack — setup, concepts, and API reference |

---

# Where A2A fits

<img src="images/image.png" width="600" alt="Where A2A fits in the stack"/>

The A2A (Agent2Agent) protocol sits at the **Agentic Orchestration** layer — the middleware that connects everything above and below it:

In [ ]:
Applications          ← User-facing apps (web, mobile, desktop)
        |
Agentic Orchestration ← ✅ A2A Protocol lives HERE
        |
Foundation Models     ← Anthropic, OpenAI, Gemini, Meta, Mistral, etc.
        |
Cloud Infrastructure  ← AWS, GCP, Azure, Snowflake
        |
Semiconductors        ← NVIDIA, AMD, Intel

**Key insights:**

- A2A is NOT a foundation model and NOT an app

- It's the **communication layer** that lets AI agents talk to each other, regardless of which model or cloud powers them

- Think of it like HTTP for the web — a standard protocol that enables interoperability

---

# A2A Architecture

<img src="images/image-1.png" width="600" alt="A2A Architecture"/>

The core architecture has two roles:

| Role | Description |

|------|-------------|

| **A2A Client** (Agent A) | The agent that initiates a request |

| **A2A Server / Remote Agent** (Agent B) | The agent that receives and handles the request |

**What is an Agent Card?**

- Agent B exposes a file at `/.well-known/agent-card.json`

- It's a self-description document — like a "business card" for the agent

- It tells Agent A: what capabilities Agent B has, what tasks it can handle, and how to communicate with it

- Agent A reads this card **before** sending any task

In [ ]:
**Flow:**
Agent A (Client) ──discovers──> Agent B's agent-card.json
Agent A (Client) ──sends task──> Agent B (A2A Server)

---

## Execution Modes

`A2A Client communicates in below modes in A2A Server`

<img src="images/image-2.png" width="600" alt="Execution Modes"/>

A2A supports 4 communication modes between Client and Server:

| Mode | Definition |

|------|-----------|

| **Synchronous** | Wait for an immediate response (blocking) |

| **Asynchronous** | Send and proceed without waiting (non-blocking) |

| **Streaming** | Receive continuous data as it's produced |

| **Push Notifications** | Server alerts the client when specific events occur |

---

### Synchronous Communication

<img src="images/image-3.png" width="550" alt="Synchronous Communication"/>

**How it works:**

- Agent A sends a **JSON-RPC** request → Agent B processes → Agent B sends a JSON-RPC response back

- Agent A **blocks and waits** until the response arrives (like a regular HTTP request/response)

**Message structure (JSON-RPC):**

- **Message** — the payload being sent

- **Role** — either `user` or `agent` (who is speaking)

- **Parts** — the actual content: `text`, `file`, or `JSON`

**When to use:**

- Short, fast tasks where you need the result before proceeding

- Example: "Translate this sentence" — quick, expect an immediate reply

**Code — what changes for Synchronous mode:**

In [ ]:
**Server** — declare `streaming=False` in `AgentCapabilities`:
# a2a_policy_agent.py — AgentCard capability flag
agent_card = AgentCard(
    ...
    capabilities=AgentCapabilities(streaming=False),  # ← sync: no SSE
    ...
)

In [ ]:
**Server — Executor** pushes ONE complete message then returns:
class PolicyAgentExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        prompt   = context.get_user_input()
        response = self.agent.answer_query(prompt)      # blocking LLM call

        # Enqueue ONE final message — framework returns it as a single HTTP response
        await event_queue.enqueue_event(new_agent_text_message(response))
        # execute() returns → connection closes → client receives the full answer

In [ ]:
**Client** — iterate responses and handle the **two possible return types**:
# client sends message, waits for one complete response
responses = client.send_message(message)

async for response in responses:
    if isinstance(response, Message):
        # Agent replied directly with a final Message (sync path)
        text = get_message_text(response)

    elif isinstance(response, tuple):
        # Framework wrapped it in a Task + Artifact (also valid in sync mode)
        task: Task       = response[0]
        artifact: Artifact = task.artifacts[0]
        text = get_message_text(artifact)

> **Why two types?** The A2A spec allows servers to reply either as a raw `Message` or as a `Task` (which wraps the result in an `Artifact`). `DefaultRequestHandler` may return either depending on its internal state — the client must handle both.

---

### Asynchronous Communication

<img src="images/image-4.png" width="550" alt="Asynchronous Communication"/>

**How it works:**

- Agent A sends a JSON-RPC message → Agent B **immediately returns a Task ID** (not the result)

- Agent A continues doing other work (non-blocking)

- Agent A **polls** (periodically checks) the Task ID to get updates

**Task object returned:**

- **ID** — unique task identifier

- **Current Status** — e.g., `submitted`, `working`, `completed`

**Result (Artifact):**

- When done, the result is an **Artifact** containing: ID + Parts (text, file, or JSON)

**When to use:**

- Long-running tasks where you can't afford to wait

- Example: "Analyze this 100-page document" — submit and check back later

**Code — what changes for Asynchronous mode:**

**Server** — `AgentCapabilities` has **no special flag** for async. Async is not a capability you declare — it is a **behaviour** determined by how the executor runs:

In [ ]:
# AgentCard — same as sync, no extra flags
agent_card = AgentCard(
    ...
    capabilities=AgentCapabilities(
        streaming=False,           # no SSE
        pushNotifications=False,   # no webhook callback
        # async is implied — if executor takes time and emits status events,
        # DefaultRequestHandler automatically returns a Task (not a Message)
        # and the client polls for completion
    ),
    ...
)

> **Why no async flag?** The A2A spec does not have an `async=True` capability. Async behaviour is inferred: if `execute()` emits a `TaskStatusUpdateEvent` before finishing, the framework knows the task is long-running and returns a `Task` object to the client. The client then polls `GET /tasks/{id}` — that is the async pattern.

In [ ]:
**Server — Executor** emits status immediately then does the long work:
class PolicyAgentExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        # Emit a status update immediately → client gets Task ID + "working" status
        await event_queue.enqueue_event(
            TaskStatusUpdateEvent(state=TaskState.working)   # ← tells client: task started
        )

        # Run the long task (could be awaited or dispatched to background)
        response = await self.agent.answer_query_async(prompt)

        # Emit the final artifact when done
        await event_queue.enqueue_event(
            TaskArtifactUpdateEvent(artifact=build_artifact(response))
        )
        # Framework marks Task as "completed" in InMemoryTaskStore

In [ ]:
**Client** — send then poll by Task ID:
# Step 1: Send message — gets back a Task ID immediately (non-blocking)
task_response = await client.send_message(message)
task_id = task_response.id          # e.g. "task-abc-123"
print(f"Task submitted: {task_id}, status: {task_response.status.state}")

# Step 2: Poll until completed
import asyncio
while True:
    task = await client.get_task(task_id)          # GET /tasks/{id}
    print(f"Status: {task.status.state}")          # submitted → working → completed
    if task.status.state in ("completed", "failed", "canceled"):
        break
    await asyncio.sleep(2)                         # wait before next poll

# Step 3: Read artifact from completed task
text = get_message_text(task.artifacts[0])

> **Key difference from sync:** The client does NOT block on `send_message`. It gets a `Task` object back immediately with `state=submitted`, then polls `get_task(id)` periodically. `InMemoryTaskStore` on the server holds the task state between requests.

---

### Streaming using Server-Side Events (SSE)

<img src="images/image-5.png" width="550" alt="Streaming SSE"/>

**How it works:**

- Agent A sends a JSON-RPC request with `streaming: true`

- Agent B **pushes updates continuously** back to Agent A as work progresses (one-directional stream from server to client)

- No need to poll — updates arrive automatically

**What gets streamed:**

- **Task Status Update Events** — e.g., "still working...", "35% complete"

- **Task Artifact Update Events** — partial results as they're generated (like streaming LLM token output)

**Key difference from async:**

- Async = you poll (pull model)

- SSE = server pushes to you automatically (push model, but over the same connection)

**When to use:**

- When you want real-time progress feedback

- Example: Streaming LLM text generation token-by-token to the user

**Code — what changes for Streaming mode:**

In [ ]:
**Server** — declare `streaming=True` and `enqueue_event` for every chunk:
# AgentCard — advertise SSE support
agent_card = AgentCard(
    ...
    capabilities=AgentCapabilities(streaming=True),   # ← enable SSE
    ...
)

class PolicyAgentExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        prompt = context.get_user_input()

        # Stream LLM tokens as they arrive — each chunk goes to the client immediately
        async for chunk in self.agent.stream_answer(prompt):   # LLM streaming call
            await event_queue.enqueue_event(
                # Partial artifact update — client receives each token/chunk in real-time
                TaskArtifactUpdateEvent(
                    artifact=build_text_artifact(chunk),
                    last_chunk=False     # ← more chunks coming
                )
            )

        # Signal stream end
        await event_queue.enqueue_event(
            TaskArtifactUpdateEvent(artifact=build_text_artifact(""), last_chunk=True)
        )

In [ ]:
**Client** — same `send_message` loop but receives events progressively:
# Client code is IDENTICAL to sync — the SDK handles SSE transparently
responses = client.send_message(message)

full_text = ""
async for response in responses:
    if isinstance(response, tuple):
        task: Task = response[0]
        if task.artifacts:
            chunk = get_message_text(task.artifacts[0])
            full_text += chunk
            print(chunk, end="", flush=True)   # ← print each token as it arrives
    elif isinstance(response, Message):
        full_text = get_message_text(response)

# full_text now has the complete response

> **Why client code looks the same:** The A2A client SDK automatically detects whether the server returned a single response or an SSE stream and normalizes both into the same `async for response in responses` loop. The difference is purely on the server side.

---

### Push Notification

<img src="images/image-6.png" width="550" alt="Push Notification"/>

**How it works:**

- Agent A sends its **Callback URL** to Agent B along with the task

- Agent A disconnects — it does NOT wait or poll

- When Agent B finishes (or an event occurs), it **calls back Agent A's URL** with a push notification

- Agent B's agent-card declares `pushNotifications: true` to indicate it supports this mode

**Key difference from SSE:**

- SSE = persistent open connection, server streams over it

- Push Notification = connection closes, server independently calls back later (like a webhook)

**When to use:**

- Long-running, fire-and-forget tasks

- When Agent A may go offline or can't maintain an open connection

- Example: "Run a nightly data pipeline, notify me when done"

**Code — what changes for Push Notification mode:**

**Full flow diagram:**

In [ ]:
CLIENT (Agent A)                              SERVER (Agent B)
      │                                              │
      │  POST /  { message, callbackUrl }            │
      │─────────────────────────────────────────────►│
      │                                              │  1. Receives callbackUrl
      │  { taskId: "task-123", state: "submitted" }  │     stores it with Task
      │◄─────────────────────────────────────────────│
      │                                              │  2. Runs long work
      │  (disconnects — does NOT wait or poll)       │     (client is gone)
                                                     │
                                                     │  3. Work completes
                                                     │
                                                     │  POST callbackUrl
                                                     │  { task: { id, status,
      │◄─────────────────────────────────────────────│    artifacts: [...] } }
      │                                              │
      │  Client's webhook receives the result        │

---

**Step 1 — Server: declare `pushNotifications=True` in AgentCapabilities**

In [ ]:
# AgentCard — advertise push notification support
agent_card = AgentCard(
    ...
    capabilities=AgentCapabilities(
        streaming=False,
        pushNotifications=True,   # ← tells clients: I will POST back to your URL
    ),
    ...
)

**Step 2 — Server Executor: do the work, enqueue result**

The executor runs normally — it does NOT explicitly call the callback URL.

`DefaultRequestHandler` reads `callbackUrl` from the original request and POSTs automatically when `execute()` completes.

In [ ]:
class PolicyAgentExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        prompt = context.get_user_input()

        # context.push_notification_config holds the callbackUrl sent by client
        # DefaultRequestHandler uses it automatically — you don't call it manually
        callback_url = context.push_notification_config.url
        print(f"Will POST result to: {callback_url}")  # e.g. http://client:8080/callback

        # Run the long work — client is NOT waiting
        response = await self.agent.run_pipeline(prompt)

        # Enqueue the result — framework handles the outbound POST to callbackUrl
        await event_queue.enqueue_event(new_agent_text_message(response))

        # What DefaultRequestHandler does internally after execute() returns:
        #   task.status = completed
        #   task.artifacts = [artifact built from enqueued events]
        #   POST callbackUrl  ← HTTP POST with full Task JSON as body

**Step 3 — How the server calls the client's URL: Handled by A2A framwork Itself**

> **You do NOT write this code.** `DefaultRequestHandler` (the A2A framework) handles the outbound POST automatically after `execute()` returns. Nothing extra to implement on the server side.

What the framework does internally (for understanding only — not your code):

In [ ]:
After execute() completes:
  1. Framework builds the final Task object
       { id, status: "completed", artifacts: [...] }
  2. Reads callbackUrl from the original request
  3. HTTP POSTs the Task JSON to callbackUrl
       POST http://client:8080/a2a/callback
       Content-Type: application/json
       Body: {
         "id": "task-123",
         "status": { "state": "completed" },
         "artifacts": [
           { "parts": [{ "type": "text", "text": "Pipeline done!" }] }
         ]
       }
  4. Done — server has no further obligation

**Your responsibility on the server = zero extra code.** Just:

- Set `AgentCapabilities(pushNotifications=True)` in AgentCard

- Enqueue the result in `execute()` as usual

**Step 4 — Client: register callback URL + expose a webhook endpoint**

In [ ]:
from fastapi import FastAPI
from a2a.types import Task, PushNotificationConfig
from a2a.utils.message import get_message_text

app = FastAPI()

CALLBACK_URL = "http://my-agent-client:8080/a2a/callback"   # must be reachable by server

# ── Part A: Send the task (fire-and-forget) ──────────────────────────────
async def send_task():
    async with httpx.AsyncClient() as httpx_client:
        client = await ClientFactory.connect(
            f"http://{host}:{port}",
            client_config=ClientConfig(httpx_client=httpx_client),
        )
        message = create_text_message_object(content="Run nightly pipeline")

        # Register callback URL — server will POST here when done
        await client.send_message(
            message,
            push_notification_config=PushNotificationConfig(url=CALLBACK_URL)
        )
        # Returns immediately with task_id — client disconnects
        print("Task submitted. Waiting for push notification...")

# ── Part B: Webhook — receives POST from server when task completes ───────
@app.post("/a2a/callback")
async def receive_push_notification(task: Task):
    # Server POSTs the full Task JSON here when work is done
    print(f"Push received! Task ID : {task.id}")
    print(f"Status       : {task.status.state}")          # "completed"

    if task.artifacts:
        result_text = get_message_text(task.artifacts[0])
        print(f"Result       : {result_text}")

    return {"status": "received"}   # acknowledge the POST

> **Key difference from async:** In async mode the client polls `GET /tasks/{id}` — it must stay alive and keep checking. In push mode the client registers a URL and the **server calls the client** when done. The client can shut down entirely between sending and receiving — the server retries the POST if needed.

---

## Summary — Choosing the Right Mode

In [ ]:
Need result immediately?          → Synchronous
Can't wait, will check later?     → Asynchronous (poll)
Want real-time updates streamed?  → SSE Streaming
Fire-and-forget + get notified?   → Push Notifications (Webhook)

> The A2A protocol's power is that **all four modes use the same JSON-RPC message format** — only the transport behavior changes, keeping agent communication consistent and interoperable.

---

## AgentCard Capabilities — Multiple Skills with Different Modes

When one agent exposes multiple skills that each use different execution modes, `AgentCapabilities` on the `AgentCard` is set to the **union** of all modes the agent supports.

In [ ]:
# AgentCard — declare union of all modes across ALL skills
agent_card = AgentCard(
    ...
    capabilities=AgentCapabilities(
        streaming=True,            # at least one skill uses SSE
        pushNotifications=True,    # at least one skill uses push
        # sync and async are always implied — no explicit flag needed
    ),
    skills=[skill_sync, skill_async, skill_sse, skill_push]
)

**Each `AgentSkill` declares its own mode:**

In [ ]:
skill_sync = AgentSkill(
    id="translate",
    name="Text Translation",
    description="Translates text — fast, returns immediately",
    tags=["translation"],
    examples=["Translate this to French"],
    # no streaming flag → defaults to sync
)

skill_sse = AgentSkill(
    id="summarize_doc",
    name="Document Summarizer",
    description="Summarizes long documents with real-time progress",
    tags=["summarization"],
    examples=["Summarize this 100-page PDF"],
    supports_streaming=True,            # ← this skill streams
)

skill_push = AgentSkill(
    id="nightly_pipeline",
    name="Data Pipeline",
    description="Runs overnight ETL — notifies on completion",
    tags=["pipeline"],
    examples=["Run nightly report"],
    supports_push_notifications=True,   # ← this skill uses push
)

**Executor routes by `context.skill_id` to apply the right behavior:**

In [ ]:
class MultiSkillExecutor(AgentExecutor):
    async def execute(self, context: RequestContext, event_queue: EventQueue) -> None:
        skill_id = context.skill_id        # A2A tells you which skill was invoked
        prompt   = context.get_user_input()

        if skill_id == "translate":
            # Sync — one complete response
            result = self.agent.translate(prompt)
            await event_queue.enqueue_event(new_agent_text_message(result))

        elif skill_id == "summarize_doc":
            # SSE — stream chunks as they arrive
            async for chunk in self.agent.stream_summary(prompt):
                await event_queue.enqueue_event(
                    TaskArtifactUpdateEvent(artifact=build_artifact(chunk), last_chunk=False)
                )
            await event_queue.enqueue_event(
                TaskArtifactUpdateEvent(artifact=build_artifact(""), last_chunk=True)
            )

        elif skill_id == "nightly_pipeline":
            # Push — server calls back client's URL when done
            result = await self.agent.run_pipeline(prompt)
            await event_queue.enqueue_event(new_agent_text_message(result))

**Rule of thumb:**

| What | Where | Rule |

|---|---|---|

| `AgentCapabilities(streaming, pushNotifications)` | `AgentCard` | Set `True` if **any** skill needs it — union of all skills |

| Per-skill mode (`supports_streaming`, etc.) | `AgentSkill` | Each skill declares its own capability |

| Routing logic | `AgentExecutor.execute()` | Branch on `context.skill_id` |

> The client fetches `AgentCard`, sees `streaming=True` and `pushNotifications=True`, and knows the agent **can** do both. It then picks the right skill by `id` when calling — the executor decides which transport behavior to apply.

---

## LangGraph Agent calling an A2A Server

This section shows how a **LangGraph agent (client side)** discovers, reasons about, and calls a remote **A2A server** — using the real `ProviderAgent` + `PolicyAgent` examples from the notebooks.

---

### How LangGraph Agent Invokes an A2A Server

> **Important distinction:**

> - **Direct A2A client call** — client already knows which server to call. It sends the message directly. No LLM involved in routing.

> - **LangGraph multi-agent** — LLM reads AgentCard descriptions of multiple servers (as tools) and decides which one to call. Explained in the next section.

This section shows the **direct call** pattern — client → one known A2A server.

In [ ]:
A2A Client (knows server URL)               A2A Server (remote)
        │                                          │
        │  1. GET /.well-known/agent.json          │
        │─────────────────────────────────────────►│
        │  AgentCard { name, skills, url }         │  (for inspection / display only)
        │◄─────────────────────────────────────────│
        │                                          │
        │  2. POST /  { message }                  │
        │─────────────────────────────────────────►│  server executor runs the skill
        │  { task / message with result }          │
        │◄─────────────────────────────────────────│

**Step 1 — Connect and fetch AgentCard (for inspection):**

In [ ]:
import httpx
from a2a.client import Client, ClientConfig, ClientFactory, create_text_message_object
from a2a.types import AgentCard, Artifact, Message, Task
from a2a.utils.message import get_message_text

async with httpx.AsyncClient(timeout=100.0) as httpx_client:

    # Connect to a KNOWN A2A server — URL is hardcoded / config-driven
    # No LLM involvement here — caller already knows which server to use
    client: Client = await ClientFactory.connect(
        "http://localhost:9997",
        client_config=ClientConfig(httpx_client=httpx_client),
    )

    # Fetch AgentCard — used for display / verification, NOT for LLM routing here
    agent_card: AgentCard = await client.get_card()
    print(agent_card.name)          # "HealthcareProviderAgent"
    print(agent_card.description)   # "An agent that can find healthcare providers..."
    for skill in agent_card.skills:
        print(skill.id, skill.description)

**Step 2 — Send message directly to the A2A server (no LLM routing):**

In [ ]:
    prompt  = "I'm based in Austin, TX. Are there any Psychiatrists near me?"
    message = create_text_message_object(content=prompt)

    # Client sends the full user message directly to the server — no LLM decides here.
    # The A2A server's executor receives it and runs the appropriate skill internally.
    responses = client.send_message(message)

    text_content = ""

    async for response in responses:
        if isinstance(response, Message):
            # Server replied with a direct Message (sync path)
            text_content = get_message_text(response)

        elif isinstance(response, tuple):
            # Server replied with Task + Artifact
            task: Task         = response[0]
            artifact: Artifact = task.artifacts[0]
            text_content       = get_message_text(artifact)

    print(text_content)   # final answer from the remote A2A agent

> **No LLM in this flow.** The client sends the message as plain text to the server. The A2A server's `AgentExecutor.execute()` is what runs — it may call an LLM internally (e.g. `ProviderAgent` uses `ChatOpenAI`) but that is inside the server, not the client. The client is just a caller.

---

### How AgentCard is Passed to LLM to Decide Which Skill to Call

> **This is the LangGraph multi-agent pattern** — where multiple A2A servers are registered as tools and the LLM decides which one to call. This is different from the direct call above.

The LangGraph agent wraps each A2A agent as a **LangChain tool**. The tool's name and description are built from the `AgentCard`. The LLM reads these at inference time and decides which tool (= which A2A server) to invoke for the user's query.

In [ ]:
AgentCard (from A2A server)
  ├── name        → tool name shown to LLM
  ├── description → tool description shown to LLM
  └── skills[]
        └── description → included in tool description

LLM sees:
  Tool: "HealthcareProviderAgent"
  Description: "Finds healthcare providers... Skills: find_healthcare_providers —
                Finds and lists healthcare providers based on location and specialty.
                Examples: Are there any Psychiatrists near me in Boston, MA?"

User query: "Find a psychiatrist in Austin TX"
  → LLM matches query to tool description → selects HealthcareProviderAgent
  → LangGraph calls the tool → tool POSTs to A2A server

**Building the LangChain tool from AgentCard:**

In [ ]:
from langchain_core.tools import tool
from a2a.client import ClientFactory, ClientConfig, create_text_message_object
from a2a.types import AgentCard, Message, Task
from a2a.utils.message import get_message_text
import httpx

def build_a2a_tool(agent_url: str):
    """
    Fetches AgentCard from the A2A server and wraps it as a LangChain tool.
    The tool name and description come directly from the AgentCard —
    this is what the LLM reads to decide whether to call this agent.
    """
    async def _call_a2a_agent(prompt: str) -> str:
        async with httpx.AsyncClient(timeout=100.0) as httpx_client:
            client = await ClientFactory.connect(
                agent_url,
                client_config=ClientConfig(httpx_client=httpx_client),
            )
            message  = create_text_message_object(content=prompt)
            responses = client.send_message(message)

            async for response in responses:
                if isinstance(response, Message):
                    return get_message_text(response)
                elif isinstance(response, tuple):
                    task: Task = response[0]
                    if task.artifacts:
                        return get_message_text(task.artifacts[0])
            return ""

    # Fetch AgentCard synchronously at setup time to read name + description
    import asyncio
    async def _get_card():
        async with httpx.AsyncClient() as httpx_client:
            client = await ClientFactory.connect(
                agent_url,
                client_config=ClientConfig(httpx_client=httpx_client),
            )
            return await client.get_card()

    card: AgentCard = asyncio.run(_get_card())

    # Build skill descriptions to include in the tool description
    skill_desc = " | ".join(
        f"{s.name}: {s.description}" for s in card.skills
    )

    # Create LangChain tool — name + description are what the LLM sees
    @tool(name_or_callable=card.name, description=f"{card.description} Skills: {skill_desc}")
    async def a2a_tool(prompt: str) -> str:
        return await _call_a2a_agent(prompt)

    return a2a_tool

---

### How LangGraph Agent Calls a Specific Skill on an A2A Server

**The LLM decides the skill** — the user's query is matched to a tool's description by the LLM. LangGraph then invokes the tool (which calls the A2A server). The A2A server's executor reads `context.skill_id` to route to the right skill internally.

**Full LangGraph agent wiring with multiple A2A agents:**

In [ ]:
from langgraph.prebuilt import create_react_agent
from langchain_openai import ChatOpenAI

# Step 1: Build tools from AgentCards of all registered A2A servers
provider_tool = build_a2a_tool("http://localhost:9997")   # HealthcareProviderAgent
policy_tool   = build_a2a_tool("http://localhost:9999")   # InsurancePolicyCoverageAgent

# Step 2: LLM sees both tool descriptions at inference time
#   Tool 1: "HealthcareProviderAgent" — Finds healthcare providers...
#   Tool 2: "InsurancePolicyCoverageAgent" — Provides insurance coverage info...

llm   = ChatOpenAI(model="gpt-4o", temperature=0)
agent = create_react_agent(
    llm,
    tools=[provider_tool, policy_tool],   # ← both A2A agents as tools
    prompt="You are a healthcare assistant. Use the available tools to answer questions."
)

# Step 3: LangGraph runs the ReAct loop
#   User query → LLM picks tool → tool POSTs to A2A server → result returned to LLM
response = await agent.ainvoke({
    "messages": [{"role": "user", "content": "I'm in Austin TX. Find a psychiatrist."}]
})
#   LLM sees: "HealthcareProviderAgent" matches → calls provider_tool
#   provider_tool: POST http://localhost:9997/ { message: "I'm in Austin TX. Find a psychiatrist." }
#   A2A server executor: runs ProviderAgent.answer_query() → returns table of doctors
#   LangGraph returns final answer to user

print(response["messages"][-1].content)

**Calling a specific skill by ID (explicit routing, no LLM decision):**

In [ ]:
from a2a.client import create_text_message_object

async with httpx.AsyncClient(timeout=100.0) as httpx_client:
    client = await ClientFactory.connect(
        "http://localhost:9997",
        client_config=ClientConfig(httpx_client=httpx_client),
    )

    message = create_text_message_object(content=prompt)

    # Pass skill_id explicitly — bypasses LLM routing, calls that skill directly
    responses = client.send_message(
        message,
        skill_id="find_healthcare_providers"   # ← specific skill on the A2A server
    )

    async for response in responses:
        if isinstance(response, Message):
            print(get_message_text(response))
        elif isinstance(response, tuple):
            task: Task = response[0]
            if task.artifacts:
                print(get_message_text(task.artifacts[0]))

**Summary — who decides what:**

| Decision | Who decides | How |

|---|---|---|

| Which A2A agent to call | LLM | Reads `AgentCard.name + description + skills` as tool description |

| Which skill within the agent | LLM (implicit) or caller (explicit) | LLM matches query to skill description; or caller sets `skill_id=` |

| How to run the skill | `AgentExecutor` on server | Branches on `context.skill_id` inside `execute()` |

| Transport (sync/async/SSE) | `AgentCapabilities` + executor behaviour | Declared in `AgentCard`, executed by `DefaultRequestHandler` |

---

## Agent Stack

**Purpose:** Explains the problem of moving an AI agent from your laptop to production, introduces **Agent Stack** as the recommended solution, and helps you choose the right deployment platform.

---

### From Local Development to Production Deployment

<img src="images/image-7.png" width="600" alt="Local to Production Deployment"/>

**The Problem:** Running an agent locally is easy. But deploying it to production requires building 6 pieces of infrastructure yourself:

| Component | What it means |

|---|---|

| **Storage & File Management** | Where does your agent store files, PDFs, embeddings? |

| **LLM Inference Integration** | How does your agent connect to the LLM API reliably at scale? |

| **Vector Database for RAG** | Where are your document embeddings stored for retrieval? |

| **Deployment Requirement** | Docker, Kubernetes, cloud config — the DevOps work |

| **Security & Authentication** | API keys, auth tokens, access control |

| **End User Facing Interface** | UI or API gateway for real users to interact with |

> Without a framework, you have to build all 6 yourself. That's the gap Agent Stack fills.

---

### Other Agentic Platform Options (and why they fall short)

<img src="images/image-8.png" width="600" alt="Agentic Platform Options"/>

Three alternatives — each with a trade-off:

| Platform | Pro | Con |

|---|---|---|

| **Framework Aligned** (e.g. LangServe) | Fully managed | Locked into one vendor AND one framework |

| **Cloud Vendor** (e.g. AWS Bedrock, Azure AI) | Framework agnostic | Vendor lock-in — hard to migrate later |

| **Custom Built** | Fits your exact needs | Heavy engineering lift — you build everything |

> None of the three gives you **all three**: freedom, flexibility, and low effort. That's the gap Agent Stack targets.

---

### What Agent Stack Is

<img src="images/image-9.png" width="600" alt="What Agent Stack Is"/>

Agent Stack is a **self-hostable, open-source infrastructure layer** for deploying agents. Its 4 key properties:

| Property | What it means for you |

|---|---|

| **Open-Source Linux Foundation Project** | Community-owned, transparent, auditable |

| **Framework Agnostic** | Works with LangGraph, CrewAI, plain Python — not tied to one library |

| **No Vendor Lock-In** | Deploy on any cloud or on-premise — switch anytime |

| **Simplified Agent Deployment** | Pre-built infra components — you write agent logic, not DevOps |

> Think of it as: *"Kubernetes for agents"* — you bring your agent, Agent Stack handles the rest.

---

### Who Is Agent Stack For?

<img src="images/image-10.png" width="600" alt="Who is Agent Stack For"/>

Two target users:

| User | Situation | Why Agent Stack helps |

|---|---|---|

| **Agent Development Teams** | Exploring multiple frameworks (LangGraph, CrewAI, A2A) | Need real infra to test agents against — not just local notebooks |

| **Platform Teams** | Managing many internal agent projects across an org | Need ONE unified deployment system instead of a custom stack per project |

> **Simple rule:** If you need *control over your data + freedom to pick any framework + fast deployment* → Agent Stack is for you.

---

### Architecture

<img src="data/agent_stack_architecture.png" width="650" alt="Agent Stack Architecture"/>

**Purpose:** Shows how Agent Stack is structured in three horizontal layers — how you package an agent, how the runtime executes it, and what backing services it relies on.

#### Layer 1 — Developer Workflow (top row, left → right)

| Step | Component | What it does |

|---|---|---|

| 1 | **Package Agents** | Use the Server SDK to wrap your agent (LangGraph, CrewAI, plain Python) into a deployable unit |

| 2 | **Deploy & Manage** | CLI to deploy, configure, and manage agents on the runtime |

| 3 | **Consume** | Auto-generated Web UI, Custom UI via Client SDK, or headless API for integrations |

#### Layer 2 — Runtime Core (middle)

- **Agent Stack Server** — the central runtime that hosts all your agents

- Self-hostable on **Kubernetes / OpenShift**

- Deployed via **Helm Charts** (standard Kubernetes packaging — one command install)

#### Layer 3 — Services (bottom, shared by all agents)

| Service | Purpose |

|---|---|

| **LLM Provider Management** | Manages API keys and connections to OpenAI, Anthropic, etc. |

| **RAG and Vector Store** | Shared embedding store for retrieval-augmented generation |

| **File Storage** | Stores agent inputs/outputs, PDFs, uploads |

| **Data Layer** | Persistent task state, conversation history |

| **Authentication** | Controls who can call which agent |

| **Secret Management** | Securely stores API keys and credentials |

> Key insight: all agents share the **same services layer** — you configure LLM keys and storage once, every agent uses them automatically.

---

# Security

**Purpose:** Shows how to add authentication between agents so only trusted callers can invoke your A2A server.

#### Image 1 — Server Side: Defining the Security Scheme (Code)

<img src="data/security_server_api_key_scheme.png" width="550" alt="Security Server API Key Scheme"/>

Three parts on the server:

In [ ]:
# 1. Define the requirement — tells clients what auth header to send
agent_card = AgentCard(
    ...
    security_scheme={"my_auth_scheme": api_key_scheme},  # declare in card
    security=[{"my_auth_scheme": []}]                    # enforce globally
)

# 2. Validate the request (the "Middleware")
class AuthContextBuilder(ServerCallContextBuilder):
    def build(self, request: Request) -> ServerCallContext:
        token = request.headers.get("x-api-key")         # read header
        if token == "super-secret":
            return ServerCallContext(user=SimpleUser("admin"))
        return ServerCallContext(user=UnauthenticatedUser())

> The `AgentCard` publicly advertises what auth is required → clients know what header to send before making a call.

#### Image 2 — Auth Flow Sequence (Agent A → Agent B)

<img src="data/security_auth_flow_sequence.png" width="500" alt="Security Auth Flow Sequence"/>

Step-by-step flow between two agents:

In [ ]:
1. Agent A  →  GET /agent-card        →  Agent B
               (discovers what auth is needed)

2. Agent A  →  Request Token (OIDC)   →  Identity Provider
               Identity Provider returns <TOKEN>

3. Agent A  →  HTTP POST /
               Authorization: Bearer <TOKEN>  →  Agent B

4. Agent B validates token:
   ├── [Token Valid]   → 200 OK  (task executes)
   └── [Token Invalid] → 403 Forbidden (rejected)

> The AgentCard discovery step (1) is what makes A2A self-describing — Agent A knows what auth to get before it even tries to call.

#### Image 3 — Client Side: Sending Credentials (Code)

<img src="data/security_client_credential_store.png" width="550" alt="Security Client Credential Store"/>

In [ ]:
# 1. Load the Credential Vault
# Map: Session ID → Schema Name → The actual secret
cred_store = InMemoryContextCredentialsStore()
await cred_store.set_credentials(
    session_id="user_session_1",
    security_scheme_name="my_auth_scheme",
    credential="super-secret"              # the API key value
)

# 2. Create Client with Auth Interceptor
# Interceptor reads the cred_store and injects the header automatically
client = await ClientFactory.connect(
    "http://localhost:8000",
    interceptors=[AuthInterceptor(cred_store)]
)

# 3. Send message — SDK auto-adds x-api-key header from cred_store
client.send_message(message, context=ClientCallContext(
    state={"sessionId": "user_session_1"}
))

> `AuthInterceptor` acts like a middleware on the client — you never manually add auth headers; the SDK handles it per session.

---

# Extension

**Purpose:** Extensions let you add **custom metadata** to an AgentCard to advertise non-standard capabilities — such as billing, rate limits, or SLA tiers — that the base A2A spec doesn't cover.

Reference: https://github.com/a2aproject/a2a-samples/tree/main/extensions

#### Image — Extension in AgentCard JSON

<img src="data/extension_agentcard_billing.png" width="550" alt="Extension AgentCard Billing"/>

In [ ]:
{
  "name": "Global Insurance Agent",
  "capabilities": { "streaming": false },
  "extensions": [
    {
      "uri": "https://example.com/ext/billing/v1",
      "description": "Defines per-query transaction fees for agent access",
      "required": true,
      "params": {
        "field_name": "x-billing-cost",
        "rate_type": "per_request",
        "currency": "USD",
        "base_fee": 0.05          ← $0.05 charged per call
      }
    }
  ],
  "skills": [
    {
      "id": "policy_check",
      "tags": ["finance", "sensitive", "billable"]
    }
  ]
}

| Field | Purpose |

|---|---|

| `uri` | Uniquely identifies the extension type (like a namespace) |

| `required: true` | Calling agent MUST support this extension to use the skill |

| `params` | Extension-specific config — here it defines billing cost per call |

> Extensions are how the A2A ecosystem grows — teams can publish their own extension specs (billing, compliance, SLA) without changing the core protocol.

---

# Registry — Why and How

## Why Use a Registry Instead of Direct Agent URLs?

In [ ]:
❌ Direct URL:  Orchestrator ──hardcoded URL──▶ Billing Agent
                Agent redeploys → orchestrator breaks

✅ Registry:   Orchestrator ──"find billing-agent"──▶ Registry ──current URL──▶ Billing Agent

| Problem | Direct URL | Registry |

|---|---|---|

| URL changes | All callers break | Registry updates once |

| New version deployed | Manual updates everywhere | Registry points to new endpoint |

| Agent goes down | Silent failure | Registry marks unhealthy |

| Discovery | Must know URL in advance | Look up by name or capability |

| Load balancing | Manual | Registry returns stable LB URL |

> Direct URLs are fine for local dev. In production multi-agent systems, always use a registry.

## Most Widely Used Registries

| Registry | Used For |

|---|---|

| **Kubernetes Service + DNS** | Most production K8s agent systems |

| **Consul** | Multi-cloud service mesh + health checks |

| **Agent Garden** (Vertex AI) | GCP A2A agent discovery |

## How Agents Notify the Registry

In [ ]:
Heartbeat (agent → registry):     POST /heartbeat every 10–30s → missed 3× = UNHEALTHY
Push on change (agent → registry): PUT /agents/billing-agent    on deploy or shutdown
Health probe (registry → agent):   GET /health every 30s        → 3 failures = removed

In GKE, Kubernetes liveness/readiness probes handle this automatically — the Service removes a pod the moment probes fail.

## Metadata an Agent Registers

In A2A, this is the **AgentCard** — the agent self-describes via `/.well-known/agent.json`:

In [ ]:
{
  "id": "billing-agent-v2",
  "url": "https://billing-agent.run.app",
  "version": "2.1.0",
  "capabilities": ["invoice_lookup", "_status"],
  "tags": ["billing", "finance", "production"],
  "health_endpoint": "/health",
  "auth": "bearer-token",
  "owner": "billing-team@company.com"
}

The AgentCard IS the registry entry — agents self-register by exposing this endpoint.

## Discovery Patterns

In [ ]:
# By capability
agent = registry.find(capability="invoice_lookup")

# By name
agent = registry.get("billing-agent")

# Agent Garden — semantic search
agents = agent_garden.search("agent that handles customer invoices")

# Kubernetes — DNS is the registry, no lookup needed
http://billing-service.default.svc.cluster.local:8080

## Load Balancing

The registry returns a **single stable URL** backed by a load balancer. Multiple instances are invisible to the caller:

In [ ]:
Caller ──▶ registry.find("billing-agent")
           returns: https://billing-agent.run.app   ← stable URL

           Behind: Load Balancer → Instance 1 / Instance 2 / Instance 3

| Platform | LB Mechanism |

|---|---|

| **Cloud Run** | Google LB auto-distributes — single URL always |

| **GKE Service** | ClusterIP round-robins across pods; DNS = discovery |

| **Agent Garden** | Agent Engine URL — GCP manages scaling behind it |

---

# Observability

**Purpose:** Shows how every agent call in a multi-agent system can be traced end-to-end through a visual tracing tool (Phoenix / Arize).

#### Image — Phoenix Trace Dashboard

<img src="data/observability_phoenix_trace.png" width="700" alt="Observability Phoenix Trace"/>

What the dashboard shows for a single user query through a multi-agent system:

In [ ]:
User query: "Analyse the risks for NVDA stock"
      │
      ▼
trading_strategy_orchestrator        ← root span (orchestrator agent)
      ├── execute_bear_risk_ag...     ← child span (Bear Risk Agent called via A2A)
      │       └── agent_run (Bear...)
      │               ├── a2a:client_1ms
      │               ├── a2a:client_2ms
      │               └── a2a:client_3ms
      └── call_llm                   ← final LLM call to compose answer

| Panel | What it shows |

|---|---|

| **Trace list (left)** | Every agent invocation with latency and status |

| **Span tree (middle)** | Nested call hierarchy — which agent called which |

| **Input / Output (right)** | Exact prompt sent and response received per span |

> This is the production-grade answer to "what happened during that slow request?" — you see every A2A hop, every LLM call, and the exact data that flowed through each step.